<a href="https://colab.research.google.com/github/aiman0642/saas-retention-intelligence/blob/main/01_data_understanding_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 — Data Understanding

**Project:** SaaS Customer Retention Intelligence System
**Dataset:** RavenStack — synthetic multi-table SaaS dataset (accounts, subscriptions,
feature_usage, support_tickets, churn_events)

**Goal of this notebook:** get to know the raw data before touching a single model.
Load every table, check shape/dtypes/missing values/duplicates, understand the
target variable and date fields, and produce a Data Dictionary.

## 1.1 Mount Google Drive

This makes every processed file this project produces persist across
sessions — mount once per session, approve the popup, and every later
module (2 through 13) will find this module's output automatically.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.2 Imports & Project Paths

In [3]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_DIR = "/content/drive/MyDrive/saas-retention-intelligence"
RAW_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"RAW_DIR       = {RAW_DIR}")
print(f"PROCESSED_DIR = {PROCESSED_DIR}")

RAW_DIR       = /content/drive/MyDrive/saas-retention-intelligence/data/raw
PROCESSED_DIR = /content/drive/MyDrive/saas-retention-intelligence/data/processed


## 1.3 Load All Tables

Five tables, linked by `account_id` / `subscription_id`:
`accounts` → `subscriptions` → `feature_usage`, and `accounts` → `support_tickets`,
`accounts` → `churn_events`.


In [4]:
accounts = pd.read_csv(f"{RAW_DIR}/ravenstack_accounts.csv")
subscriptions = pd.read_csv(f"{RAW_DIR}/ravenstack_subscriptions.csv")
feature_usage = pd.read_csv(f"{RAW_DIR}/ravenstack_feature_usage.csv")
support_tickets = pd.read_csv(f"{RAW_DIR}/ravenstack_support_tickets.csv")
churn_events = pd.read_csv(f"{RAW_DIR}/ravenstack_churn_events.csv")

tables = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events,
}

for name, df in tables.items():
    print(f"{name:<18} shape={df.shape}")

accounts           shape=(500, 10)
subscriptions      shape=(5000, 14)
feature_usage      shape=(25000, 8)
support_tickets    shape=(2000, 9)
churn_events       shape=(600, 9)


## 1.4 Shape, Dtypes & First Look

One pass per table: dtypes, head, and a `.info()` summary.

In [5]:
for name, df in tables.items():
    print("=" * 70)
    print(name.upper())
    print("=" * 70)
    print(df.dtypes)
    print()
    display(df.head(3))

ACCOUNTS
account_id         object
account_name       object
industry           object
country            object
signup_date        object
referral_source    object
plan_tier          object
seats               int64
is_trial             bool
churn_flag           bool
dtype: object



,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False


SUBSCRIPTIONS
subscription_id      object
account_id           object
start_date           object
end_date             object
plan_tier            object
seats                 int64
mrr_amount            int64
arr_amount            int64
is_trial               bool
upgrade_flag           bool
downgrade_flag         bool
churn_flag             bool
billing_frequency    object
auto_renew_flag        bool
dtype: object



,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False


FEATURE_USAGE
usage_id               object
subscription_id        object
usage_date             object
feature_name           object
usage_count             int64
usage_duration_secs     int64
error_count             int64
is_beta_feature          bool
dtype: object



,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False


SUPPORT_TICKETS
ticket_id                       object
account_id                      object
submitted_at                    object
closed_at                       object
resolution_time_hours          float64
priority                        object
first_response_time_minutes      int64
satisfaction_score             float64
escalation_flag                   bool
dtype: object



,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False


CHURN_EVENTS
churn_event_id               object
account_id                   object
churn_date                   object
reason_code                  object
refund_amount_usd           float64
preceding_upgrade_flag         bool
preceding_downgrade_flag       bool
is_reactivation                bool
feedback_text                object
dtype: object



,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features


## 1.5 Missing Values

Check nulls per table. The README flags that `satisfaction_score`,
`feature_usage` fields, and `churn feedback` intentionally contain nulls —
so some missingness here is expected, not a data quality bug.

In [6]:
for name, df in tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"--- {name} ---")
    if missing.empty:
        print("No missing values.")
    else:
        print(missing.to_frame("missing_count").assign(
            pct=lambda x: (x["missing_count"] / len(df) * 100).round(1)
        ))
    print()

--- accounts ---
No missing values.

--- subscriptions ---
          missing_count   pct
end_date           4514  90.3

--- feature_usage ---
No missing values.

--- support_tickets ---
                    missing_count   pct
satisfaction_score            825  41.2

--- churn_events ---
               missing_count   pct
feedback_text            148  24.7



## 1.6 Duplicate Rows & Key Uniqueness

Check for fully duplicated rows, and confirm each table's primary key is
actually unique (important before any joins in later modules).

In [7]:
key_cols = {
    "accounts": "account_id",
    "subscriptions": "subscription_id",
    "feature_usage": "usage_id",
    "support_tickets": "ticket_id",
    "churn_events": "churn_event_id",
}

for name, df in tables.items():
    dup_rows = df.duplicated().sum()
    key = key_cols[name]
    dup_keys = df[key].duplicated().sum()
    print(f"{name:<18} duplicate rows: {dup_rows:<5} duplicate '{key}': {dup_keys}")

accounts           duplicate rows: 0     duplicate 'account_id': 0
subscriptions      duplicate rows: 0     duplicate 'subscription_id': 0
feature_usage      duplicate rows: 0     duplicate 'usage_id': 21
support_tickets    duplicate rows: 0     duplicate 'ticket_id': 0
churn_events       duplicate rows: 0     duplicate 'churn_event_id': 0


## 1.7 Referential Integrity Check

Confirm foreign keys actually resolve — every `account_id` referenced in the
child tables should exist in `accounts`, and every `subscription_id` in
`feature_usage` should exist in `subscriptions`.

In [8]:
account_ids = set(accounts["account_id"])

for name, df in [("subscriptions", subscriptions), ("support_tickets", support_tickets), ("churn_events", churn_events)]:
    orphans = (~df["account_id"].isin(account_ids)).sum()
    print(f"{name:<18} orphan account_id rows: {orphans}")

subscription_ids = set(subscriptions["subscription_id"])
orphan_usage = (~feature_usage["subscription_id"].isin(subscription_ids)).sum()
print(f"{'feature_usage':<18} orphan subscription_id rows: {orphan_usage}")

subscriptions      orphan account_id rows: 0
support_tickets    orphan account_id rows: 0
churn_events       orphan account_id rows: 0
feature_usage      orphan subscription_id rows: 0


## 1.8 Date Fields — Parse & Sanity-Check

Convert every date/datetime column to actual `datetime64`, and check the
overall date range each table covers. The README claims validated temporal
logic (signup ≤ subscription ≤ churn) — verify that holds.

In [9]:
accounts["signup_date"] = pd.to_datetime(accounts["signup_date"])
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])
feature_usage["usage_date"] = pd.to_datetime(feature_usage["usage_date"])
support_tickets["submitted_at"] = pd.to_datetime(support_tickets["submitted_at"])
support_tickets["closed_at"] = pd.to_datetime(support_tickets["closed_at"])
churn_events["churn_date"] = pd.to_datetime(churn_events["churn_date"])

date_cols = {
    "accounts.signup_date": accounts["signup_date"],
    "subscriptions.start_date": subscriptions["start_date"],
    "subscriptions.end_date": subscriptions["end_date"],
    "feature_usage.usage_date": feature_usage["usage_date"],
    "support_tickets.submitted_at": support_tickets["submitted_at"],
    "churn_events.churn_date": churn_events["churn_date"],
}

for label, series in date_cols.items():
    print(f"{label:<32} min={series.min()}  max={series.max()}")

accounts.signup_date             min=2023-01-02 00:00:00  max=2024-12-31 00:00:00
subscriptions.start_date         min=2023-01-09 00:00:00  max=2024-12-31 00:00:00
subscriptions.end_date           min=2023-04-05 00:00:00  max=2024-12-31 00:00:00
feature_usage.usage_date         min=2023-01-01 00:00:00  max=2024-12-31 00:00:00
support_tickets.submitted_at     min=2023-01-02 00:00:00  max=2024-12-31 00:00:00
churn_events.churn_date          min=2023-01-25 00:00:00  max=2024-12-31 00:00:00


## 1.9 Target Variable — Understanding Churn

There are two churn signals in this dataset: `churn_flag` (boolean, on
`accounts` and `subscriptions`) and the `churn_events` table (one row per
churn *instance*, with reason codes and feedback).

In [10]:
print("accounts.churn_flag value counts:")
print(accounts["churn_flag"].value_counts(normalize=True).round(3))
print()
print("subscriptions.churn_flag value counts:")
print(subscriptions["churn_flag"].value_counts(normalize=True).round(3))
print()
print(f"Accounts with at least one churn_event: {churn_events['account_id'].nunique()} / {accounts['account_id'].nunique()}")
print()
print("churn_events.reason_code distribution:")
print(churn_events["reason_code"].value_counts())

accounts.churn_flag value counts:
churn_flag
False    0.78
True     0.22
Name: proportion, dtype: float64

subscriptions.churn_flag value counts:
churn_flag
False    0.903
True     0.097
Name: proportion, dtype: float64

Accounts with at least one churn_event: 352 / 500

churn_events.reason_code distribution:
reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64


## 1.10 Numerical vs Categorical Feature Identification

Split each table's columns into numerical, categorical, boolean, ID, and
date buckets — this feeds directly into the Data Dictionary and later
guides encoding/scaling decisions in Module 4.

In [11]:
def classify_columns(df, id_cols=(), date_cols=()):
    result = {"id": [], "date": [], "boolean": [], "numerical": [], "categorical": []}
    for col in df.columns:
        if col in id_cols:
            result["id"].append(col)
        elif col in date_cols:
            result["date"].append(col)
        elif df[col].dtype == bool:
            result["boolean"].append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            result["numerical"].append(col)
        else:
            result["categorical"].append(col)
    return result

classification = {
    "accounts": classify_columns(accounts, id_cols=["account_id"], date_cols=["signup_date"]),
    "subscriptions": classify_columns(subscriptions, id_cols=["subscription_id", "account_id"], date_cols=["start_date", "end_date"]),
    "feature_usage": classify_columns(feature_usage, id_cols=["usage_id", "subscription_id"], date_cols=["usage_date"]),
    "support_tickets": classify_columns(support_tickets, id_cols=["ticket_id", "account_id"], date_cols=["submitted_at", "closed_at"]),
    "churn_events": classify_columns(churn_events, id_cols=["churn_event_id", "account_id"], date_cols=["churn_date"]),
}

for table, buckets in classification.items():
    print(f"--- {table} ---")
    for bucket, cols in buckets.items():
        if cols:
            print(f"  {bucket:<12}: {cols}")
    print()

--- accounts ---
  id          : ['account_id']
  date        : ['signup_date']
  boolean     : ['is_trial', 'churn_flag']
  numerical   : ['seats']
  categorical : ['account_name', 'industry', 'country', 'referral_source', 'plan_tier']

--- subscriptions ---
  id          : ['subscription_id', 'account_id']
  date        : ['start_date', 'end_date']
  boolean     : ['is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'auto_renew_flag']
  numerical   : ['seats', 'mrr_amount', 'arr_amount']
  categorical : ['plan_tier', 'billing_frequency']

--- feature_usage ---
  id          : ['usage_id', 'subscription_id']
  date        : ['usage_date']
  boolean     : ['is_beta_feature']
  numerical   : ['usage_count', 'usage_duration_secs', 'error_count']
  categorical : ['feature_name']

--- support_tickets ---
  id          : ['ticket_id', 'account_id']
  date        : ['submitted_at', 'closed_at']
  boolean     : ['escalation_flag']
  numerical   : ['resolution_time_hours', 'first_respo

## 1.11 Data Dictionary

Consolidated reference table — every column, its table, type bucket, and a
short description (from the dataset's own README). This is the deliverable
for Module 1 and gets referenced throughout the rest of the project.

In [12]:
data_dictionary = pd.DataFrame([
    # accounts
    ("accounts", "account_id", "id", "Unique customer (primary key)"),
    ("accounts", "account_name", "categorical", "Fictional company name"),
    ("accounts", "industry", "categorical", "SaaS vertical (e.g., DevTools, EdTech)"),
    ("accounts", "country", "categorical", "ISO-2 country code"),
    ("accounts", "signup_date", "date", "Account creation date"),
    ("accounts", "referral_source", "categorical", "organic, ads, event, partner, other"),
    ("accounts", "plan_tier", "categorical", "Initial plan (Basic, Pro, Enterprise)"),
    ("accounts", "seats", "numerical", "Licensed user count"),
    ("accounts", "is_trial", "boolean", "Currently trialing"),
    ("accounts", "churn_flag", "boolean", "Churned at any point"),
    # subscriptions
    ("subscriptions", "subscription_id", "id", "Unique subscription (primary key)"),
    ("subscriptions", "account_id", "id (FK)", "Links to accounts.account_id"),
    ("subscriptions", "start_date", "date", "Subscription start"),
    ("subscriptions", "end_date", "date", "Nullable for active plans"),
    ("subscriptions", "plan_tier", "categorical", "Plan at time of billing"),
    ("subscriptions", "seats", "numerical", "Licensed seats"),
    ("subscriptions", "mrr_amount", "numerical", "Monthly recurring revenue"),
    ("subscriptions", "arr_amount", "numerical", "Annual recurring revenue"),
    ("subscriptions", "is_trial", "boolean", "Trial status"),
    ("subscriptions", "upgrade_flag", "boolean", "Plan upgraded mid-cycle"),
    ("subscriptions", "downgrade_flag", "boolean", "Plan downgraded mid-cycle"),
    ("subscriptions", "churn_flag", "boolean", "True if subscription ended"),
    ("subscriptions", "billing_frequency", "categorical", "monthly or annual"),
    ("subscriptions", "auto_renew_flag", "boolean", "Auto-renew enabled (~80% true)"),
    # feature_usage
    ("feature_usage", "usage_id", "id", "Unique usage event"),
    ("feature_usage", "subscription_id", "id (FK)", "Links to subscriptions.subscription_id"),
    ("feature_usage", "usage_date", "date", "Date of usage"),
    ("feature_usage", "feature_name", "categorical", "One of 40 SaaS features"),
    ("feature_usage", "usage_count", "numerical", "Event frequency"),
    ("feature_usage", "usage_duration_secs", "numerical", "Time spent"),
    ("feature_usage", "error_count", "numerical", "Logged errors"),
    ("feature_usage", "is_beta_feature", "boolean", "~10% flagged as beta"),
    # support_tickets
    ("support_tickets", "ticket_id", "id", "Unique ticket"),
    ("support_tickets", "account_id", "id (FK)", "Links to accounts.account_id"),
    ("support_tickets", "submitted_at", "date", "Time opened"),
    ("support_tickets", "closed_at", "date", "Time resolved"),
    ("support_tickets", "resolution_time_hours", "numerical", "Duration to resolve"),
    ("support_tickets", "priority", "categorical", "low, medium, high, urgent"),
    ("support_tickets", "first_response_time_minutes", "numerical", "Minutes to first response"),
    ("support_tickets", "satisfaction_score", "numerical", "1-5, null = no response"),
    ("support_tickets", "escalation_flag", "boolean", "True if escalated"),
    # churn_events
    ("churn_events", "churn_event_id", "id", "Unique churn instance"),
    ("churn_events", "account_id", "id (FK)", "Links to accounts.account_id"),
    ("churn_events", "churn_date", "date", "When account left"),
    ("churn_events", "reason_code", "categorical", "pricing, support, features, etc."),
    ("churn_events", "refund_amount_usd", "numerical", "$0 default, ~25% have credit/refund"),
    ("churn_events", "preceding_upgrade_flag", "boolean", "Had upgrade within 90 days"),
    ("churn_events", "preceding_downgrade_flag", "boolean", "Had downgrade within 90 days"),
    ("churn_events", "is_reactivation", "boolean", "~10% were previously churned"),
    ("churn_events", "feedback_text", "categorical", "Optional customer comment"),
], columns=["table", "column", "type", "description"])

display(data_dictionary)

,table,column,type,description
0,accounts,account_id,id,Unique customer (primary key)
1,accounts,account_name,categorical,Fictional company name
2,accounts,industry,categorical,"SaaS vertical (e.g., DevTools, EdTech)"
3,accounts,country,categorical,ISO-2 country code
4,accounts,signup_date,date,Account creation date
5,accounts,referral_source,categorical,"organic, ads, event, partner, other"
6,accounts,plan_tier,categorical,"Initial plan (Basic, Pro, Enterprise)"
7,accounts,seats,numerical,Licensed user count
8,accounts,is_trial,boolean,Currently trialing
9,accounts,churn_flag,boolean,Churned at any point


## 1.12 Save Cleaned/Typed Tables

Persist the date-parsed versions to `data/processed/` so Module 2 (EDA)
and Module 3 (Cohort Analysis) can load ready-to-use tables instead of
re-parsing dates every time.

In [13]:
# PROCESSED_DIR is under Google Drive — these files persist across sessions.
accounts.to_csv(f"{PROCESSED_DIR}/accounts.csv", index=False)
subscriptions.to_csv(f"{PROCESSED_DIR}/subscriptions.csv", index=False)
feature_usage.to_csv(f"{PROCESSED_DIR}/feature_usage.csv", index=False)
support_tickets.to_csv(f"{PROCESSED_DIR}/support_tickets.csv", index=False)
churn_events.to_csv(f"{PROCESSED_DIR}/churn_events.csv", index=False)
data_dictionary.to_csv(f"{PROCESSED_DIR}/data_dictionary.csv", index=False)

print(f"Saved typed tables + data dictionary to {PROCESSED_DIR}")

Saved typed tables + data dictionary to /content/drive/MyDrive/saas-retention-intelligence/data/processed


## 1.13 Module 1 Summary

- 5 tables loaded: accounts (500), subscriptions (5,000), feature_usage
  (25,000), support_tickets (2,000), churn_events (600)
- Referential integrity confirmed — no orphan foreign keys
- Two churn signals identified: `churn_flag` (account/subscription level)
  and `churn_events` (event-level, with reason codes)
- Nulls are expected in `satisfaction_score` and `feedback_text` — not a
  data quality issue
- Data Dictionary built and saved

**Next:** Module 2 — Exploratory Data Analysis